# 20-S · Stage 3 V6-S — VideoMamba-S K400, two-stage adaptation

목표는 **v5-D acceleration 설계를 유지한 채 빠른 VideoMamba-S backbone으로 V6 recipe를 검증**하는 것이다.

이 notebook은 VideoMamba-M MASK 실험을 대체/덮어쓰지 않는다. 별도 run directory를 사용한다.

### Stage A — interface adaptation
- 1 epoch
- 4,000 micro-steps = 1,000 optimizer updates
- VideoMamba-S **전체 freeze + eval**
- 학습: `SpatialMomentPooler`, `head.project`, BiGRU/기존 heads, accel-state head

### Stage B — last-2 fine-tuning
- 5 epochs
- epoch당 4,000 micro-steps = 1,000 optimizer updates
- VideoMamba-S layer **22/23** + final norm unfreeze
- 마지막 2 Mamba layer는 activation checkpointing
- 나머지 backbone은 계속 freeze + eval

공통:
- batch 2 × grad accumulation 4 = effective batch 8
- input `32 × 288 × 384`
- v5-D data/sampler/loss/auxiliary design 고정
- balanced DataLoader 안정화 설정 유지
- inference lock: stride 8 / center floor .25 / accel weight 0 / state weight 0
- 최종 checkpoint 선택은 **Stage B acceleration proxy 중심**

### Backbone 변경
- `videomamba_middle` → `videomamba_small`
- embed dim `576 → 384`
- depth `32 → 24`
- 공식 checkpoint: `videomamba_s16_k400_f32_res224.pth`
- 공식 K400-S 32f recipe의 `drop_path=0.35` 사용
- 공식 VideoMamba model zoo에는 S MASK checkpoint가 없으므로 **ImageNet-1K → K400 S checkpoint**를 사용


## 0. Setup

In [ ]:
from __future__ import annotations

import copy
import gc
import json
import math
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

from google.colab import drive

try:
    drive.mount("/content/drive", force_remount=False)
except Exception:
    drive.mount("/content/drive", force_remount=True)

REPO = Path("/content/Blackbox-Detection")
REPO_URL = "https://github.com/sangchun1/Blackbox-Detection.git"
BRANCH = "stage3-sangchun"

if not (REPO / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH,
         "--single-branch", REPO_URL, str(REPO)],
        check=True,
    )
else:
    current = subprocess.run(
        ["git", "-C", str(REPO), "branch", "--show-current"],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if current != BRANCH:
        subprocess.run(
            ["git", "-C", str(REPO), "checkout", BRANCH],
            check=True,
        )
    dirty = subprocess.run(
        ["git", "-C", str(REPO), "status", "--porcelain"],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if not dirty:
        subprocess.run(
            ["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH],
            check=True,
        )
    else:
        print("WARNING: local repo dirty; git pull skipped")

src_target = (
    REPO
    / "src/blackbox_detection/stage3/v6a_videomamba.py"
)
cfg_target = (
    REPO
    / "configs/stage3/videomamba_m_mask_can_v6a.yaml"
)

if not src_target.is_file():
    raise FileNotFoundError(src_target)

if not cfg_target.is_file():
    raise FileNotFoundError(cfg_target)

print("V6-A source:", src_target)
print("V6-A config:", cfg_target)

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade-strategy", "only-if-needed",
        "timm==1.0.15",
        "fvcore==0.1.5.post20221221",
        "iopath==0.1.10",
        "yacs==0.1.8",
        "einops==0.8.1",
        "wandb==0.29.0",
        "easydict==1.13",
        "huggingface-hub==0.34.4",
        "ninja",
        "packaging",
        "wheel",
    ],
    check=True,
)

SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd
import torch
import yaml
import site
import cv2

# Stage-3 performs thousands of repeated MP4 decodes. Keep OpenCV from
# creating its own CPU thread pool on top of the DataLoader/runtime.
cv2.setNumThreads(0)
if hasattr(cv2, "ocl"):
    cv2.ocl.setUseOpenCL(False)

from blackbox_detection.stage3.proxy_metrics import assert_dacon_metric_contract
from blackbox_detection.utils import (
    dataloader_seed_kwargs,
    finish_wandb,
    init_wandb,
    seed_everything,
)

DRIVE_ROOT = Path("/content/drive/MyDrive/Blackbox-Detection")
DATA_ROOT = DRIVE_ROOT / "DATASET"
COMMA_ROOT = DATA_ROOT / "comma2k19" / "processed" / "v1"
A2D2_ROOT = DATA_ROOT / "A2D2" / "processed"
MANIFEST_ROOT = DRIVE_ROOT / "manifests/stage3/v1"
OUTPUT_ROOT = DRIVE_ROOT / "outputs/stage3"
PRETRAINED_ROOT = DRIVE_ROOT / "pretrained"
WANDB_KEY_PATH = DRIVE_ROOT / "wandb_key.txt"

LOCAL_PRETRAINED_ROOT = Path("/content/pretrained")
LOCAL_OUTPUT_ROOT = Path("/content/stage3_runs")
for p in [OUTPUT_ROOT, PRETRAINED_ROOT, LOCAL_PRETRAINED_ROOT, LOCAL_OUTPUT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

CFG_PATH = cfg_target
cfg = yaml.safe_load(CFG_PATH.read_text(encoding="utf-8"))

# ------------------------------------------------------------------
# V6-S experiment override
# ------------------------------------------------------------------
# Reuse the audited V6-A config for data/loss/training/inference, but swap
# only the VideoMamba backbone family and isolate all outputs from the M run.
cfg["experiment"]["name"] = "videomamba_s_k400_can_v6s_backbone_ablation"

_s_model = cfg["model"]
_s_model["backbone"] = "videomamba_small"
_s_model["official_checkpoint"] = "videomamba_s16_k400_f32_res224.pth"
_s_model["official_pretraining"] = "ImageNet-1K_then_Kinetics-400"
_s_model["official_input_frames"] = 32
_s_model["official_resolution"] = 224
_s_model["embed_dim"] = 384
_s_model["depth"] = 24
_s_model["official_drop_path_rate"] = 0.35

# The downstream interface remains 384-d exactly as in V6-A.
assert int(_s_model["feature_dim"]) == 384

cfg.setdefault("logging", {})["wandb_group"] = "videomamba_s_k400_can_v6s"

# Runtime-stability A/B result: W&B is not required for training artifacts and
# was ruled out as the primary crash cause. Keep it hard-disabled in this
# notebook so a repository YAML change cannot silently re-enable it.
cfg.setdefault("logging", {})["wandb_enabled"] = False

stats = json.loads(
    (MANIFEST_ROOT / "target_stats.json").read_text(encoding="utf-8")
)

SEED = int(cfg["seed"])
seed_everything(SEED, deterministic=False)
assert_dacon_metric_contract()

GIT_COMMIT = subprocess.run(
    ["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()

loss_cfg = copy.deepcopy(cfg["loss"])
loss_cfg["base"]["normalization"] = {
    name: {
        "mean": float(stats[name]["mean"]),
        "std": float(stats[name]["std"]),
    }
    for name in ("speed_mps", "accel_from_speed_mps2")
}

print("Python         :", sys.version)
print("Torch          :", torch.__version__)
print("CUDA           :", torch.version.cuda)
print("GPU            :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("Git            :", BRANCH, GIT_COMMIT)
print("Config         :", CFG_PATH)
print("metric contract: PASS")
print("W&B forced OFF :", not bool(cfg.get("logging", {}).get("wandb_enabled", True)))
print("OpenCV threads :", cv2.getNumThreads())


## 1. Pin official VideoMamba + persistent BiMamba wheel cache

공식 VideoMamba는 저자 fork의 `Mamba(..., bimamba=True)`와 CUDA extension을 사용한다.

**새 Colab runtime마다 10분씩 다시 컴파일하지 않도록**, 첫 성공 빌드의
`causal_conv1d` / `mamba_ssm` wheel을 Google Drive에 영구 캐시한다.

캐시 key에는 다음을 포함한다.
- pinned VideoMamba commit
- Python version
- PyTorch version
- CUDA version
- C++ ABI
- CPU architecture

동작:
1. 현재 runtime에서 이미 정상 import되면 즉시 통과
2. Drive wheel cache가 있으면 wheel만 설치
3. cache miss일 때만 CUDA extension을 소스에서 컴파일
4. 생성된 wheel을 Drive에 저장
5. 현재 Colab kernel에서도 import + 실제 CUDA forward smoke

따라서 같은 환경의 다음 새 runtime부터는 소스 컴파일을 반복하지 않는다.

In [ ]:
import importlib
import platform
import site

VIDEO_MAMBA_REPO = Path("/content/VideoMamba")
vm_cfg = cfg["model"]
VIDEO_MAMBA_COMMIT = str(vm_cfg["official_commit"])

if not (VIDEO_MAMBA_REPO / ".git").is_dir():
    subprocess.run(
        [
            "git", "clone", "-q",
            "https://github.com/OpenGVLab/VideoMamba.git",
            str(VIDEO_MAMBA_REPO),
        ],
        check=True,
    )

subprocess.run(
    [
        "git", "-C", str(VIDEO_MAMBA_REPO),
        "fetch", "--all", "--tags",
    ],
    check=True,
)
subprocess.run(
    [
        "git", "-C", str(VIDEO_MAMBA_REPO),
        "checkout", "-q", VIDEO_MAMBA_COMMIT,
    ],
    check=True,
)

ACTUAL_VM_COMMIT = subprocess.run(
    ["git", "-C", str(VIDEO_MAMBA_REPO), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
assert ACTUAL_VM_COMMIT == VIDEO_MAMBA_COMMIT

VM_CAUSAL_ROOT = VIDEO_MAMBA_REPO / "causal-conv1d"
VM_MAMBA_ROOT = VIDEO_MAMBA_REPO / "mamba"

for required in (
    VM_CAUSAL_ROOT / "setup.py",
    VM_MAMBA_ROOT / "setup.py",
    VM_MAMBA_ROOT / "mamba_ssm",
):
    if not required.exists():
        raise FileNotFoundError(required)


def _fresh_python_bimamba_smoke() -> bool:
    probe = subprocess.run(
        [
            sys.executable,
            "-c",
            (
                "import torch; "
                "import causal_conv1d_cuda; "
                "import selective_scan_cuda; "
                "from mamba_ssm.modules.mamba_simple import Mamba; "
                "m=Mamba(d_model=16, expand=2, bimamba=True); "
                "print('fresh-process BiMamba:', type(m).__name__)"
            ),
        ],
        capture_output=True,
        text=True,
    )
    if probe.returncode != 0:
        print("Fresh-process BiMamba probe failed:")
        print(probe.stderr[-3000:])
        return False
    print(probe.stdout.strip())
    return True


# ------------------------------------------------------------------
# Persistent wheel cache
# ------------------------------------------------------------------
# Compiling the official CUDA extensions can take ~10 minutes on a fresh
# Colab runtime. Cache the built wheels on Google Drive, keyed by the
# environment that determines binary compatibility.
python_tag = f"py{sys.version_info.major}{sys.version_info.minor}"
torch_tag = str(torch.__version__).split("+")[0].replace(".", "_")
cuda_tag = str(torch.version.cuda or "cpu").replace(".", "_")
abi_tag = f"abi{int(torch._C._GLIBCXX_USE_CXX11_ABI)}"
arch_tag = platform.machine().replace("/", "_")
commit_tag = ACTUAL_VM_COMMIT[:12]

WHEEL_CACHE_DIR = (
    DRIVE_ROOT
    / "cache"
    / "videomamba_wheels"
    / (
        f"{commit_tag}_{python_tag}_torch{torch_tag}_"
        f"cu{cuda_tag}_{abi_tag}_{arch_tag}"
    )
)
WHEEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_WHEEL_DIR = Path("/content/videomamba_wheels")
LOCAL_WHEEL_DIR.mkdir(parents=True, exist_ok=True)

print("VideoMamba wheel cache:", WHEEL_CACHE_DIR)


def _cached_wheel(prefix: str):
    wheels = sorted(
        WHEEL_CACHE_DIR.glob(f"{prefix}-*.whl"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    return wheels[0] if wheels else None


def _local_wheel(prefix: str):
    wheels = sorted(
        LOCAL_WHEEL_DIR.glob(f"{prefix}-*.whl"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    return wheels[0] if wheels else None


def _pip_install_wheel(wheel: Path):
    print("install wheel:", wheel.name)
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--force-reinstall",
            "--no-deps",
            str(wheel),
        ],
        check=True,
    )


def _build_wheel(package_dir: Path, expected_prefix: str) -> Path:
    # Remove old local wheel with the same package prefix so we know which
    # artifact came from this build.
    for old in LOCAL_WHEEL_DIR.glob(f"{expected_prefix}-*.whl"):
        old.unlink()

    env = dict(os.environ)
    env["MAX_JOBS"] = "4"

    print("BUILDING CUDA wheel (first time only):", package_dir)
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "wheel",
            "--no-build-isolation",
            "--no-deps",
            "-w",
            str(LOCAL_WHEEL_DIR),
            str(package_dir),
        ],
        check=True,
        env=env,
    )

    wheel = _local_wheel(expected_prefix)
    if wheel is None:
        raise RuntimeError(
            f"wheel build finished but {expected_prefix}-*.whl was not produced"
        )
    return wheel


if not _fresh_python_bimamba_smoke():
    cached_causal = _cached_wheel("causal_conv1d")
    cached_mamba = _cached_wheel("mamba_ssm")

    if cached_causal is not None and cached_mamba is not None:
        print("Persistent VideoMamba CUDA wheel cache: HIT")
        _pip_install_wheel(cached_causal)
        _pip_install_wheel(cached_mamba)
    else:
        print("Persistent VideoMamba CUDA wheel cache: MISS")
        print(
            "This runtime will compile the two official CUDA extensions once. "
            "The resulting wheels will be saved to Google Drive for future runtimes."
        )

        causal_wheel = _build_wheel(
            VM_CAUSAL_ROOT,
            "causal_conv1d",
        )
        # mamba's build/import expects causal-conv1d to be available.
        _pip_install_wheel(causal_wheel)

        mamba_wheel = _build_wheel(
            VM_MAMBA_ROOT,
            "mamba_ssm",
        )

        for wheel in (causal_wheel, mamba_wheel):
            cached = WHEEL_CACHE_DIR / wheel.name
            shutil.copy2(wheel, cached)
            print("cached:", cached)

        _pip_install_wheel(mamba_wheel)

    if not _fresh_python_bimamba_smoke():
        raise RuntimeError(
            "Official VideoMamba CUDA extensions are still unusable after "
            "wheel-cache install/build. Do not continue."
        )
else:
    print("VideoMamba CUDA extensions already available in this runtime.")


# ------------------------------------------------------------------
# Make the installation visible to THIS running Colab kernel.
# ------------------------------------------------------------------
for sp in site.getsitepackages():
    site.addsitedir(sp)

# The Python sources are pinned to the same VideoMamba commit. Adding the
# roots explicitly avoids the editable-install/.pth visibility issue in an
# already-running Colab kernel. The compiled .so files come from site-packages.
for source_root in (
    str(VM_MAMBA_ROOT),
    str(VM_CAUSAL_ROOT),
):
    if source_root not in sys.path:
        sys.path.insert(0, source_root)

importlib.invalidate_caches()

try:
    import causal_conv1d
    import causal_conv1d_cuda
    import selective_scan_cuda
    import mamba_ssm
    from mamba_ssm.modules.mamba_simple import Mamba
except Exception as exc:
    raise RuntimeError(
        "VideoMamba dependencies are installed but unavailable in the "
        "current Colab kernel."
    ) from exc

print("current-kernel mamba_ssm :", mamba_ssm.__file__)
print("current-kernel causal    :", causal_conv1d.__file__)
print("selective_scan_cuda      :", selective_scan_cuda.__file__)
print("causal_conv1d_cuda       :", causal_conv1d_cuda.__file__)

if not torch.cuda.is_available():
    raise RuntimeError("V6-A VideoMamba training requires a CUDA GPU runtime")

_smoke_mamba = Mamba(
    d_model=16,
    expand=2,
    bimamba=True,
).cuda().eval()

_smoke_x = torch.randn(
    1, 8, 16,
    device="cuda",
    dtype=torch.float32,
)

with torch.inference_mode():
    _smoke_y = _smoke_mamba(_smoke_x)

assert _smoke_y.shape == _smoke_x.shape
assert torch.isfinite(_smoke_y).all()

print(
    "CURRENT-KERNEL BIMAMBA CUDA SMOKE: PASS",
    tuple(_smoke_y.shape),
)

del _smoke_mamba, _smoke_x, _smoke_y
torch.cuda.empty_cache()

print("VideoMamba commit:", ACTUAL_VM_COMMIT)

## 2. Download official VideoMamba-S K400 32-frame checkpoint

선정 이유:
- **32 frames**: 현재 Stage3 `T=32`와 정확히 일치
- **Small / 384-d / 24 layers**: M보다 훨씬 가벼워 반복 실험과 안정성 확보에 유리
- 공식 VideoMamba model zoo의 K400 32f S checkpoint 사용
- 공식 S MASK checkpoint는 제공되지 않으므로 standard S(ImageNet-1K → K400)를 사용

공식 checkpoint는 224×224에서 fine-tune되었지만, V6 dense adapter가 positional
embedding을 실제 Stage3의 18×24 patch grid(288×384)에 bicubic interpolation한다.


In [ ]:
from huggingface_hub import hf_hub_download

def _is_usable_file(path: Path, min_bytes: int = 1) -> bool:
    try:
        return path.is_file() and path.stat().st_size >= int(min_bytes)
    except OSError:
        return False

def copy_file_to_local(
    source: Path,
    destination: Path,
    *,
    min_bytes: int = 1,
):
    destination.parent.mkdir(parents=True, exist_ok=True)
    tmp = destination.with_name(destination.name + ".copy.tmp")
    tmp.unlink(missing_ok=True)
    with source.open("rb") as src, tmp.open("wb") as dst:
        shutil.copyfileobj(src, dst, length=16 * 1024 * 1024)
    if tmp.stat().st_size < int(min_bytes):
        raise OSError(f"staged file too small: {tmp.stat().st_size}")
    os.replace(tmp, destination)

VM_CKPT_NAME = str(vm_cfg["official_checkpoint"])
VM_CKPT_DRIVE = PRETRAINED_ROOT / VM_CKPT_NAME
VM_CKPT_LOCAL = LOCAL_PRETRAINED_ROOT / VM_CKPT_NAME

if not _is_usable_file(VM_CKPT_DRIVE, 10_000_000):
    downloaded = Path(
        hf_hub_download(
            repo_id=str(vm_cfg["official_checkpoint_repo"]),
            filename=VM_CKPT_NAME,
        )
    )
    copy_file_to_local(
        downloaded,
        VM_CKPT_DRIVE,
        min_bytes=10_000_000,
    )

if not _is_usable_file(VM_CKPT_LOCAL, 10_000_000):
    copy_file_to_local(
        VM_CKPT_DRIVE,
        VM_CKPT_LOCAL,
        min_bytes=10_000_000,
    )

print("VideoMamba checkpoint:", VM_CKPT_LOCAL)
print("size MiB:", VM_CKPT_LOCAL.stat().st_size / 2**20)


## 3. Build the exact V5-D data/sampler protocol

event cache도 v5-D와 동일한 `v5d/event_index_t32_s16.npz`를 재사용한다.

In [ ]:
from torch.utils.data import DataLoader, Subset

from blackbox_detection.stage3.v5_dataset import build_event_balanced_sampler
from blackbox_detection.stage3.v5d_dataset import MixedStage3CANV5DDataset

dc = cfg["data"]
tc = cfg["training"]
source_cfg = dc["sources"]

a2d2_manifest = pd.read_csv(A2D2_ROOT / "manifest.csv")
assert int(a2d2_manifest["num_frames"].sum()) == 9151

train_sources = [
    {
        "name": "comma2k19",
        "manifest": MANIFEST_ROOT / "comma_train.csv",
        "processed_root": COMMA_ROOT,
        **source_cfg["comma2k19"],
    },
    {
        "name": "a2d2",
        "manifest": a2d2_manifest,
        "processed_root": A2D2_ROOT,
        **source_cfg["a2d2"],
    },
]
val_sources = [
    {
        "name": "comma2k19",
        "manifest": MANIFEST_ROOT / "comma_val_id.csv",
        "processed_root": COMMA_ROOT,
        **source_cfg["comma2k19"],
    },
]

train_ds = MixedStage3CANV5DDataset(
    train_sources,
    target_stats=stats,
    clip_len=dc["clip_len"],
    window_stride=dc["train_window_stride"],
    input_size=(dc["input_height"], dc["input_width"]),
    random_flip=dc["train_random_flip"],
    seed=SEED,
)
val_ds = MixedStage3CANV5DDataset(
    val_sources,
    target_stats=stats,
    clip_len=dc["clip_len"],
    window_stride=dc["val_window_stride"],
    input_size=(dc["input_height"], dc["input_width"]),
    random_flip=False,
    seed=SEED + 1,
)

sampler_cfg = dc["sampler"]
EVENT_CACHE = (
    DRIVE_ROOT / "manifests/stage3/v5d"
    / f"event_index_t{dc['clip_len']}_s{dc['train_window_stride']}.npz"
)

train_sampler, sampler_report = build_event_balanced_sampler(
    train_ds,
    num_samples=int(tc["max_steps_per_epoch"]) * int(tc["batch_size"]),
    event_multipliers=sampler_cfg["event_multipliers"],
    source_multipliers=sampler_cfg["source_multipliers"],
    inverse_frequency_power=sampler_cfg["inverse_frequency_power"],
    max_normalized_weight=sampler_cfg["max_normalized_weight"],
    seed=SEED,
    cache_path=EVENT_CACHE,
)

n_val_eval = min(int(tc["max_val_steps"]), len(val_ds))
val_indices = np.linspace(
    0, len(val_ds) - 1,
    num=n_val_eval,
    dtype=np.int64,
)
val_eval_ds = Subset(
    val_ds,
    np.unique(val_indices).tolist(),
)

# ============================================================
# Balanced DataLoader stability mode
# ============================================================
# Stage 3 clips are much larger than Stage 1 samples, so keep only one
# background decoder worker and a single prefetched batch. This restores
# decode/compute overlap without returning to the original long-lived, pinned
# two-worker queue that repeatedly crashed during Stage A.
#
# `spawn` avoids forking a parent process after CUDA / VideoMamba custom CUDA
# extensions have already been initialized. No model, sampler, loss, metric,
# batch-size, or effective-batch setting is changed.
BALANCED_DATALOADER = True

if BALANCED_DATALOADER:
    train_workers = 1
    val_workers = 0
    pin_memory = False
else:
    train_workers = int(dc["num_workers"])
    val_workers = int(dc.get("val_num_workers", 0))
    pin_memory = True

train_loader_kwargs = dict(
    dataset=train_ds,
    batch_size=tc["batch_size"],
    sampler=train_sampler,
    shuffle=False,
    num_workers=train_workers,
    pin_memory=pin_memory,
    persistent_workers=False,
    **dataloader_seed_kwargs(SEED),
)
if train_workers > 0:
    train_loader_kwargs.update(
        prefetch_factor=1,
        multiprocessing_context="spawn",
    )
train_loader = DataLoader(**train_loader_kwargs)

val_loader = DataLoader(
    val_eval_ds,
    batch_size=tc["batch_size"],
    shuffle=False,
    num_workers=val_workers,
    pin_memory=pin_memory,
    persistent_workers=False,
    **dataloader_seed_kwargs(SEED + 1),
)

assert len(val_eval_ds) == int(tc["max_val_steps"])
print("event cache          :", EVENT_CACHE)
print("train windows        :", len(train_ds))
print("sampled train / epoch:", len(train_sampler))
print("validation windows   :", len(val_eval_ds))
print("A2D2 expected share  :", sampler_report["expected_source_share"]["a2d2"])
print("BALANCED_DATALOADER  :", BALANCED_DATALOADER)
print("train workers        :", train_workers)
print("val workers          :", val_workers)
print("pin memory           :", pin_memory)
print("prefetch factor      :", 1 if train_workers > 0 else None)
print("mp context           :", "spawn" if train_workers > 0 else None)


## 4. Construct VideoMamba-S and load official K400 weights

공식 K400 S 32-frame script와 맞춰 `videomamba_small`, 32 frames, tubelet=1,
`drop_path=0.35`로 architecture를 만든다. Classification head는 downstream에서
사용하지 않지만 checkpoint integrity 확인을 위해 400-class head까지 load한다.


In [ ]:
import importlib.util

VIDEO_SM = VIDEO_MAMBA_REPO / "videomamba/video_sm"
if str(VIDEO_SM) not in sys.path:
    sys.path.insert(0, str(VIDEO_SM))

vm_file = VIDEO_SM / "models/videomamba.py"
spec = importlib.util.spec_from_file_location(
    "official_videomamba_model",
    vm_file,
)
if spec is None or spec.loader is None:
    raise ImportError(f"could not create import spec for {vm_file}")

module_name = spec.name
sys.modules.pop(module_name, None)

vm_module = importlib.util.module_from_spec(spec)
sys.modules[module_name] = vm_module

try:
    spec.loader.exec_module(vm_module)
except Exception:
    sys.modules.pop(module_name, None)
    raise

official_backbone = vm_module.videomamba_small(
    pretrained=False,
    img_size=int(vm_cfg["official_resolution"]),
    num_classes=400,
    drop_path_rate=float(vm_cfg["official_drop_path_rate"]),
    kernel_size=int(vm_cfg["tubelet_size"]),
    num_frames=int(vm_cfg["official_input_frames"]),
    use_checkpoint=False,
    checkpoint_num=0,
)

raw = torch.load(
    VM_CKPT_LOCAL,
    map_location="cpu",
    weights_only=False,
)
if isinstance(raw, dict):
    state = None
    for key in ("model", "module", "state_dict"):
        if isinstance(raw.get(key), dict):
            state = dict(raw[key])
            break
    if state is None:
        state = dict(raw)
else:
    raise TypeError(type(raw))

state = {
    (k[7:] if k.startswith("module.") else k): v
    for k, v in state.items()
}

msg = official_backbone.load_state_dict(state, strict=False)
allowed_missing = set()
allowed_unexpected = set()
if set(msg.missing_keys) - allowed_missing:
    raise RuntimeError(f"official checkpoint missing keys: {msg.missing_keys}")
if set(msg.unexpected_keys) - allowed_unexpected:
    raise RuntimeError(f"official checkpoint unexpected keys: {msg.unexpected_keys}")

# load_state_dict has copied the checkpoint tensors into the module. Keeping
# both the original checkpoint payload and normalized state dict alive wastes
# CPU RAM for the whole run.
del raw
del state
gc.collect()

print("official checkpoint load: PASS")
print("released checkpoint CPU state: PASS")
print("embed dim:", official_backbone.embed_dim)
print("depth    :", len(official_backbone.layers))
print("drop path:", vm_cfg["official_drop_path_rate"])
assert int(official_backbone.embed_dim) == int(vm_cfg["embed_dim"]) == 384
assert len(official_backbone.layers) == int(vm_cfg["depth"]) == 24


## 5. Build V6-S + V5-D compatible warm start

VideoMamba-S는 384-d backbone이다. 아래 backbone-dependent 부분은 fresh로 둔다.

- VideoMamba-S backbone
- 384-d `SpatialMomentPooler`
- backbone-dependent `head.project`

아래는 V5-D E2에서 warm-start한다.

- BiGRU temporal stack
- continuous CAN heads
- acceleration ordinal/fusion
- STOP / turn / steering / brake / throttle auxiliary heads
- V5-D 3-state acceleration head

Stage A와 Stage B는 **서로 다른 checkpoint/optimizer directory**를 사용한다.
또한 V6-S run name은 M run과 다르므로 기존 M checkpoint를 절대 resume하지 않는다.


In [ ]:
from blackbox_detection.stage3.v6a_videomamba import (
    VideoMambaDenseCANV6A,
    load_v5d_compatible_warm_start,
    split_v6a_optimizer_parameters,
)
from blackbox_detection.stage3.trainer import build_scheduler
from blackbox_detection.stage3.v5d_trainer import V5DTrainer

mc = cfg["model"]
tc = cfg["training"]
two_stage = tc["two_stage"]
stage_a_cfg = dict(two_stage["stage_a"])
stage_b_cfg = dict(two_stage["stage_b"])

fusion_cfg = dict(mc["accel_fusion"])
spatial_cfg = dict(mc["spatial_pool"])
state_cfg = dict(mc["accel_state"])

model = VideoMambaDenseCANV6A(
    official_backbone,
    feature_dim=mc["feature_dim"],
    temporal_hidden=mc["temporal_hidden"],
    temporal_layers=mc["temporal_layers"],
    spatial_grid=tuple(spatial_cfg["grid"]),
    spatial_gate_init=spatial_cfg["gate_init"],
    accel_ordinal_thresholds_mps2=mc["accel_ordinal_thresholds_mps2"],
    accel_fusion_enabled=fusion_cfg["enabled"],
    accel_fusion_hidden=fusion_cfg["hidden"],
    accel_fusion_gate_init=fusion_cfg["gate_init"],
    accel_fusion_detach_ordinal_inputs=fusion_cfg["detach_ordinal_inputs"],
    accel_state_thresholds_mps2=state_cfg["thresholds_mps2"],
    accel_state_hidden=state_cfg["hidden"],
    accel_state_dropout=state_cfg["dropout"],
    stop_thresholds_mps=mc["stop_thresholds_mps"],
    turn_yaw_thresholds_rps=mc["turn_yaw_thresholds_rps"],
    steer_activity_thresholds=mc["steer_activity_thresholds"],
    brake_thresholds_bar=mc["brake_thresholds_bar"],
    throttle_thresholds_pct=mc["throttle_thresholds_pct"],
)

RUN_VARIANT = cfg["experiment"]["name"]
RUN_DIR = OUTPUT_ROOT / RUN_VARIANT
LOCAL_RUN_DIR = LOCAL_OUTPUT_ROOT / RUN_VARIANT

STAGE_A_DIR = RUN_DIR / "stage_a_interface"
LOCAL_STAGE_A_DIR = LOCAL_RUN_DIR / "stage_a_interface"

for p in (RUN_DIR, LOCAL_RUN_DIR, STAGE_A_DIR, LOCAL_STAGE_A_DIR):
    p.mkdir(parents=True, exist_ok=True)

def restore_persistent_checkpoints(persistent_dir, local_dir):
    for filename in (
        "latest.pt",
        "best.pt",
        "best_proxy.pt",
        "best_accel_proxy.pt",
        "history.csv",
    ):
        persistent = persistent_dir / filename
        local = local_dir / filename
        if (
            not local.is_file()
            and persistent.is_file()
        ):
            min_bytes = 1_000_000 if filename.endswith(".pt") else 1
            copy_file_to_local(
                persistent,
                local,
                min_bytes=min_bytes,
            )

restore_persistent_checkpoints(STAGE_A_DIR, LOCAL_STAGE_A_DIR)
restore_persistent_checkpoints(RUN_DIR, LOCAL_RUN_DIR)

stage_a_resume = LOCAL_STAGE_A_DIR / "latest.pt"
stage_b_resume = LOCAL_RUN_DIR / "latest.pt"

warm_report = None
if not stage_a_resume.is_file() and not stage_b_resume.is_file():
    wc = cfg["warm_start"]
    source_drive = OUTPUT_ROOT / wc["run_name"] / wc["checkpoint"]
    source_local = (
        LOCAL_PRETRAINED_ROOT
        / f"{wc['run_name']}__{wc['checkpoint']}"
    )
    if not _is_usable_file(source_local, 1_000_000):
        if not _is_usable_file(source_drive, 1_000_000):
            raise FileNotFoundError(source_drive)
        copy_file_to_local(
            source_drive,
            source_local,
            min_bytes=1_000_000,
        )

    warm_report = load_v5d_compatible_warm_start(
        model,
        source_local,
    )
    if warm_report["unexpected_after_partial_load"]:
        raise RuntimeError(
            warm_report["unexpected_after_partial_load"]
        )

    print("V5-D compatible warm start: PASS")
    print("source epoch  :", warm_report["checkpoint_epoch"])
    print("copied tensors:", warm_report["copied_tensors"])
else:
    print("Existing V6-A checkpoint found; V5-D one-time warm start skipped.")

frozen_report = model.configure_frozen_backbone()
assert frozen_report.depth == int(vm_cfg["depth"])
assert frozen_report.trainable_layer_indices == ()
assert frozen_report.trainable_backbone_params == 0

print("Stage A backbone config:", frozen_report)


## 6. Phase-specific optimizer builders

두 단계는 optimizer와 scheduler를 완전히 분리한다.

- Stage A: interface + downstream heads만 optimizer에 포함
- Stage B: 위 항목 + **마지막 2 Mamba layer(22/23)** + final norm 포함

두 단계 모두 effective batch는 8이며 각 epoch에 1,000 optimizer updates다.


In [ ]:
def add_decay_groups(
    out,
    *,
    family_name,
    named_params,
    lr,
    weight_decay,
):
    decay, no_decay = [], []
    for name, p in named_params:
        if p.ndim <= 1 or name.endswith(".bias"):
            no_decay.append(p)
        else:
            decay.append(p)

    if decay:
        out.append({
            "params": decay,
            "lr": float(lr),
            "weight_decay": float(weight_decay),
            "name": f"{family_name}_decay",
        })
    if no_decay:
        out.append({
            "params": no_decay,
            "lr": float(lr),
            "weight_decay": 0.0,
            "name": f"{family_name}_no_decay",
        })


def build_phase_optimizer_and_scheduler(
    model,
    *,
    phase: str,
    phase_cfg: dict,
):
    is_stage_b = phase == "stage_b"
    families = split_v6a_optimizer_parameters(
        model,
        require_backbone=is_stage_b,
    )

    groups = []
    add_decay_groups(
        groups,
        family_name="interface",
        named_params=families["interface"],
        lr=tc["interface_learning_rate"],
        weight_decay=tc["interface_weight_decay"],
    )
    add_decay_groups(
        groups,
        family_name="head",
        named_params=families["head"],
        lr=tc["head_learning_rate"],
        weight_decay=tc["head_weight_decay"],
    )
    add_decay_groups(
        groups,
        family_name="accel_state",
        named_params=families["accel_state_head"],
        lr=tc["accel_state_head_learning_rate"],
        weight_decay=tc["accel_state_head_weight_decay"],
    )

    if is_stage_b:
        backbone_depth = len(model.backbone.backbone.layers)
        penultimate_idx = backbone_depth - 2
        last_idx = backbone_depth - 1
        add_decay_groups(
            groups,
            family_name=f"mamba{penultimate_idx}",
            named_params=families["backbone_penultimate"],
            lr=tc["penultimate_layer_learning_rate"],
            weight_decay=tc["backbone_weight_decay"],
        )
        add_decay_groups(
            groups,
            family_name=f"mamba{last_idx}",
            named_params=families["backbone_last"],
            lr=tc["last_layer_learning_rate"],
            weight_decay=tc["backbone_weight_decay"],
        )
        add_decay_groups(
            groups,
            family_name="mamba_norm",
            named_params=families["backbone_final_norm"],
            lr=tc["final_norm_learning_rate"],
            weight_decay=tc["backbone_weight_decay"],
        )

    optimizer = torch.optim.AdamW(groups)

    micro_steps_per_epoch = min(
        len(train_loader),
        int(phase_cfg["max_steps_per_epoch"]),
    )
    optimizer_steps_per_epoch = max(
        math.ceil(
            micro_steps_per_epoch
            / int(tc["grad_accum_steps"])
        ),
        1,
    )
    total_steps = (
        optimizer_steps_per_epoch
        * int(phase_cfg["epochs"])
    )

    scheduler = build_scheduler(
        optimizer,
        total_steps=total_steps,
        warmup_ratio=tc["warmup_ratio"],
        min_ratio=tc["min_learning_rate_ratio"],
    )
    return optimizer, scheduler, optimizer_steps_per_epoch


def print_optimizer(optimizer, *, phase):
    print(f"\n[{phase}]")
    for group in optimizer.param_groups:
        print(
            group["name"],
            "lr=", group["lr"],
            "wd=", group["weight_decay"],
            "params=", sum(p.numel() for p in group["params"]),
        )


## 7. W&B

Stage A와 Stage B를 하나의 W&B run에 기록하되 checkpoint는 분리한다.

In [ ]:
lc = cfg.get("logging", {})
# Keep W&B disabled for this stability run regardless of repository YAML.
WANDB_ENABLED = False
wandb_run = None
print("W&B enabled:", WANDB_ENABLED)

if WANDB_ENABLED:
    import wandb

    if not WANDB_KEY_PATH.is_file():
        raise FileNotFoundError(WANDB_KEY_PATH)
    wandb.login(
        key=WANDB_KEY_PATH.read_text(encoding="utf-8").strip(),
        relogin=False,
    )
    finish_wandb()

    run_id_path = RUN_DIR / "wandb_run_id.txt"
    stored_run_id = (
        run_id_path.read_text(encoding="utf-8").strip()
        if run_id_path.is_file()
        else None
    )
    stored_run_id = stored_run_id or None

    wandb_run = init_wandb(
        project=str(lc.get("wandb_project", "blackbox-stage3")),
        entity=os.getenv("WANDB_ENTITY") or None,
        name=f"{RUN_VARIANT}__seed{SEED}__{GIT_COMMIT}",
        group=str(lc.get("wandb_group", "videomamba_s_k400_can_v6s")),
        tags=[
            "stage3",
            "v6a",
            "videomamba-small",
            "imagenet-k400-pretrain",
            "two-stage",
            "interface-adaptation",
            "last2-finetune",
            "acceleration",
            "a2d2",
            "t32",
            "event-balanced",
            "kinematics",
        ],
        run_id=stored_run_id,
        resume="allow",
        config={
            "git_commit": GIT_COMMIT,
            "videomamba_commit": ACTUAL_VM_COMMIT,
            "run_variant": RUN_VARIANT,
            "seed": SEED,
            "two_stage": two_stage,
            "frozen_report": frozen_report.__dict__,
            "sampler_report": sampler_report,
            "warm_start_report": warm_report,
        },
    )
    if not stored_run_id and wandb_run is not None:
        run_id_path.write_text(
            str(wandb_run.id),
            encoding="utf-8",
        )

vc = cfg["validation"]
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
AMP_DTYPE = (
    torch.bfloat16
    if str(tc["amp_dtype"]).lower() in {"bf16", "bfloat16"}
    else torch.float16
)
model.to(DEVICE)

print("device:", DEVICE)
print("two-stage schedule:", json.dumps(two_stage, indent=2))


## 8. Full-resolution two-stage smoke

실제 입력 `1×3×32×288×384`으로 두 단계를 각각 backward한다.

### Stage A 확인
- VideoMamba backbone 전체 gradient 없음
- `SpatialMomentPooler`, `head.project`, accel-state head에는 gradient 존재

### Stage B 확인
- 마지막 trainable tail 직전 layer(21) gradient 없음
- 마지막 2 layer(22/23) gradient 존재
- fresh interface/accel-state head도 계속 gradient 존재

layer index는 backbone depth에서 동적으로 계산하므로 S/M 모두 같은 검증 코드를 사용할 수 있다.
이 셀에서 OOM이면 본 학습을 시작하지 않는다.


In [ ]:
from blackbox_detection.stage3.v5d_accel import v5d_multitask_loss


def grad_norm(module):
    grads = [
        p.grad.detach().float().norm()
        for p in module.parameters()
        if p.grad is not None
    ]
    return (
        float(torch.stack(grads).norm().cpu())
        if grads
        else 0.0
    )


smoke = next(iter(train_loader))

video = smoke["video"][:1].to(DEVICE)
target = smoke["target"][:1].to(DEVICE)
valid = smoke["valid"][:1].to(DEVICE)
aux = {
    k: v[:1].to(DEVICE)
    for k, v in smoke["aux"].items()
}


# ============================================================
# Stage A smoke
# ============================================================
model.configure_frozen_backbone()
model.zero_grad(set_to_none=True)

opt_a, _, _ = build_phase_optimizer_and_scheduler(
    model,
    phase="stage_a",
    phase_cfg=stage_a_cfg,
)

opt_a.zero_grad(set_to_none=True)
model.train()

assert model.backbone.backbone.training is False
assert not any(
    p.requires_grad
    for p in model.backbone.backbone.parameters()
)

with torch.autocast(
    device_type=DEVICE.type,
    dtype=AMP_DTYPE,
    enabled=DEVICE.type == "cuda",
):
    out_a = model(video)
    loss_a, _ = v5d_multitask_loss(
        out_a, target, valid, aux, loss_cfg, stats=stats,
    )

assert torch.isfinite(loss_a)
loss_a.backward()

assert all(
    p.grad is None
    for p in model.backbone.backbone.parameters()
)

stage_a_grads = {
    "spatial": grad_norm(model.spatial_pool),
    "project": grad_norm(model.head.project),
    "state": grad_norm(model.accel_state_head),
}
print("Stage A grad norms:", stage_a_grads)
for name, value in stage_a_grads.items():
    assert value > 0 and np.isfinite(value), (name, value)

print("Stage A full-res backward: PASS")
model.zero_grad(set_to_none=True)
del opt_a, out_a, loss_a


# ============================================================
# Stage B smoke
# - Dynamically select the last N layers (S: 22/23, M: 30/31).
# ============================================================
ft_report = model.configure_partial_backbone(
    last_n_layers=int(stage_b_cfg["last_n_layers"]),
    train_final_norm=bool(stage_b_cfg["train_final_norm"]),
    checkpoint_trainable=bool(stage_b_cfg["checkpoint_trainable"]),
)

backbone = model.backbone.backbone
layers = backbone.layers
depth = len(layers)
tail_n = int(stage_b_cfg["last_n_layers"])
trainable_indices = tuple(range(depth - tail_n, depth))
prefix_last_idx = depth - tail_n - 1

assert ft_report.depth == int(vm_cfg["depth"]) == depth
assert ft_report.trainable_layer_indices == trainable_indices
assert prefix_last_idx >= 0

model.zero_grad(set_to_none=True)
stale = [
    name
    for name, p in backbone.named_parameters()
    if p.grad is not None
]
print("backbone params with stale grad:", stale[:20])
print("stale grad count:", len(stale))
assert len(stale) == 0

opt_b, _, _ = build_phase_optimizer_and_scheduler(
    model,
    phase="stage_b",
    phase_cfg=stage_b_cfg,
)
opt_b.zero_grad(set_to_none=True)

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

model.train()

assert backbone.training is False
assert layers[prefix_last_idx].training is False
assert not any(p.requires_grad for p in layers[prefix_last_idx].parameters())
for idx in trainable_indices:
    assert layers[idx].training is True
    assert all(p.requires_grad for p in layers[idx].parameters())

# Disable stochastic depth ONLY for this connectivity smoke.
for idx in trainable_indices:
    layers[idx].drop_path.eval()
backbone.drop_path.eval()

for idx in trainable_indices:
    assert layers[idx].drop_path.training is False
assert backbone.drop_path.training is False

print(
    "Stage B smoke DropPath probabilities:",
    {
        **{
            f"layer{idx}": float(getattr(layers[idx].drop_path, "drop_prob", 0.0))
            for idx in trainable_indices
        },
        "final": float(getattr(backbone.drop_path, "drop_prob", 0.0)),
    },
)

with torch.autocast(
    device_type=DEVICE.type,
    dtype=AMP_DTYPE,
    enabled=DEVICE.type == "cuda",
):
    out_b = model(video)
    loss_b, _ = v5d_multitask_loss(
        out_b, target, valid, aux, loss_cfg, stats=stats,
    )

assert torch.isfinite(loss_b)
assert tuple(out_b["accel_state_logits"].shape) == (
    1, dc["clip_len"], 4, 3,
)
loss_b.backward()

assert all(
    p.grad is None
    for p in layers[prefix_last_idx].parameters()
)

stage_b_grads = {
    **{f"layer{idx}": grad_norm(layers[idx]) for idx in trainable_indices},
    "final_norm": grad_norm(backbone.norm_f),
    "spatial": grad_norm(model.spatial_pool),
    "project": grad_norm(model.head.project),
    "state": grad_norm(model.accel_state_head),
}

print("Stage B grad norms:")
for name, value in stage_b_grads.items():
    print(f"  {name:12s}: {value:.6e}")
    assert value > 0 and np.isfinite(value), (name, value)

peak_gib = (
    torch.cuda.max_memory_allocated() / 2**30
    if DEVICE.type == "cuda"
    else 0.0
)
print("Stage B full-res backward: PASS")
print("Stage B peak GPU GiB:", peak_gib)

# Restore normal Stage-B training state.
model.train()
assert backbone.training is False
assert layers[prefix_last_idx].training is False
for idx in trainable_indices:
    assert layers[idx].training is True
    assert layers[idx].drop_path.training is True
assert backbone.drop_path.training is False

model.zero_grad(set_to_none=True)

del (
    opt_b, video, target, valid, aux, out_b, loss_b, smoke,
)
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

print("smoke host/GPU cleanup: PASS")


## 9. Load fixed v5-D baseline diagnostics

V6-A의 목표는 Stage3 전체가 아니라 **acceleration이 backbone 교체로 개선되는지**다.
따라서 동일 800-window protocol의 v5-D E2 summary를 baseline으로 고정한다.

In [ ]:
BASELINE_RUN = OUTPUT_ROOT / cfg["experiment"]["baseline_run"]
BASELINE_SUMMARY = BASELINE_RUN / "summary.json"
if not BASELINE_SUMMARY.is_file():
    raise FileNotFoundError(BASELINE_SUMMARY)

baseline = json.loads(
    BASELINE_SUMMARY.read_text(encoding="utf-8")
)

baseline_metrics = {
    "stage3": float(baseline["diagnostic_best_proxy_stage3"]),
    "accel": float(baseline["diagnostic_best_proxy_accel"]),
    "steer": float(baseline["diagnostic_best_proxy_steer"]),
    "medium_accelerating": float(
        baseline["best_medium_f1_accelerating"]
    ),
    "medium_dynamic_to_constant": float(
        baseline["best_medium_dynamic_to_constant_rate"]
    ),
}
print(json.dumps(baseline_metrics, indent=2))


## 10. Train / resume — Stage A → Stage B

Stage A와 B의 checkpoint directory가 분리되어 있으므로 resume이 안전하다.

- Stage A: `.../stage_a_interface/latest.pt`
- Stage B: `.../latest.pt`

Stage A가 이미 끝났다면 checkpoint를 load하고 즉시 Stage B로 넘어간다.
Stage B가 중단된 경우에는 Stage B `latest.pt`에서 이어간다.

In [ ]:
def make_trainer(
    *,
    optimizer,
    scheduler,
    local_dir,
    sync_dir,
    phase_name,
):
    phase_config = copy.deepcopy(cfg)
    phase_config["training_phase"] = phase_name
    return V5DTrainer(
        model,
        optimizer,
        scheduler=scheduler,
        device=DEVICE,
        grad_accum_steps=tc["grad_accum_steps"],
        grad_clip_norm=tc["grad_clip_norm"],
        amp_dtype=tc["amp_dtype"],
        loss_weights=loss_cfg,
        stats=stats,
        proxy_rules=vc["proxy_rules"],
        output_dir=local_dir,
        sync_dir=sync_dir,
        wandb_enabled=WANDB_ENABLED,
        log_interval=tc["log_interval"],
        config=phase_config,
    )


# ============================================================
# Stage A: frozen VideoMamba, interface adaptation
# ============================================================
stage_a_resume = LOCAL_STAGE_A_DIR / "latest.pt"
stage_b_resume = LOCAL_RUN_DIR / "latest.pt"

if stage_b_resume.is_file():
    print(
        "Stage B checkpoint already exists; "
        "Stage A execution is skipped."
    )
    history_a = []
    stage_a_history_csv = LOCAL_STAGE_A_DIR / "history.csv"
    if stage_a_history_csv.is_file():
        stage_a_flat = pd.read_csv(stage_a_history_csv)
        print(
            "Recovered Stage A history rows:",
            len(stage_a_flat),
        )
else:
    model.configure_frozen_backbone()

    optimizer_a, scheduler_a, updates_a = (
        build_phase_optimizer_and_scheduler(
            model,
            phase="stage_a",
            phase_cfg=stage_a_cfg,
        )
    )
    print_optimizer(optimizer_a, phase="Stage A")
    print(
        "Stage A optimizer updates/epoch:",
        updates_a,
    )

    trainer_a = make_trainer(
        optimizer=optimizer_a,
        scheduler=scheduler_a,
        local_dir=LOCAL_STAGE_A_DIR,
        sync_dir=STAGE_A_DIR,
        phase_name="stage_a_interface_adaptation",
    )

    history_a = trainer_a.fit(
        train_loader,
        val_loader,
        epochs=int(stage_a_cfg["epochs"]),
        max_train_steps=int(
            stage_a_cfg["max_steps_per_epoch"]
        ),
        max_val_steps=None,
        resume_from=(
            stage_a_resume
            if stage_a_resume.is_file()
            else None
        ),
        early_stopping_patience=int(
            stage_a_cfg["early_stopping_patience"]
        ),
        backfill_validation_on_resume=False,
    )

    if not (LOCAL_STAGE_A_DIR / "latest.pt").is_file():
        raise RuntimeError(
            "Stage A did not produce latest.pt"
        )

    print("Stage A COMPLETE")

# ============================================================
# Stage B: unfreeze last 2 Mamba layers + final norm
# ============================================================
ft_report = model.configure_partial_backbone(
    last_n_layers=int(stage_b_cfg["last_n_layers"]),
    train_final_norm=bool(stage_b_cfg["train_final_norm"]),
    checkpoint_trainable=bool(
        stage_b_cfg["checkpoint_trainable"]
    ),
)
expected_tail = tuple(
    range(
        int(vm_cfg["depth"]) - int(stage_b_cfg["last_n_layers"]),
        int(vm_cfg["depth"]),
    )
)
assert ft_report.trainable_layer_indices == expected_tail

optimizer_b, scheduler_b, updates_b = (
    build_phase_optimizer_and_scheduler(
        model,
        phase="stage_b",
        phase_cfg=stage_b_cfg,
    )
)
print_optimizer(optimizer_b, phase="Stage B")
print("Stage B optimizer updates/epoch:", updates_b)

trainer_b = make_trainer(
    optimizer=optimizer_b,
    scheduler=scheduler_b,
    local_dir=LOCAL_RUN_DIR,
    sync_dir=RUN_DIR,
    phase_name="stage_b_last2_finetune",
)

stage_b_resume = LOCAL_RUN_DIR / "latest.pt"
if (
    not stage_b_resume.is_file()
    and history_a
):
    trainer_b.global_step = int(
        history_a[-1].get("global_step") or 0
    )

history_b = trainer_b.fit(
    train_loader,
    val_loader,
    epochs=int(stage_b_cfg["epochs"]),
    max_train_steps=int(
        stage_b_cfg["max_steps_per_epoch"]
    ),
    max_val_steps=None,
    resume_from=(
        stage_b_resume
        if stage_b_resume.is_file()
        else None
    ),
    early_stopping_patience=int(
        stage_b_cfg["early_stopping_patience"]
    ),
    backfill_validation_on_resume=False,
)

def flatten_phase_history(history, phase):
    rows = []
    for row in history:
        rows.append({
            "phase": phase,
            "epoch": row["epoch"],
            "minutes": row["minutes"],
            "learning_rate": row.get("learning_rate"),
            "global_step": row.get("global_step"),
            "max_gpu_memory_gib": row.get(
                "max_gpu_memory_gib"
            ),
            **{
                f"train/{k}": v
                for k, v in row["train"].items()
            },
            **{
                f"val/{k}": v
                for k, v in row["val"].items()
            },
        })
    return rows

combined_rows = []
if history_a:
    combined_rows += flatten_phase_history(
        history_a,
        "stage_a",
    )
else:
    stage_a_csv = LOCAL_STAGE_A_DIR / "history.csv"
    if stage_a_csv.is_file():
        a_df = pd.read_csv(stage_a_csv)
        a_df.insert(0, "phase", "stage_a")
        combined_rows += a_df.to_dict("records")

combined_rows += flatten_phase_history(
    history_b,
    "stage_b",
)
combined_history_df = pd.DataFrame(combined_rows)
display(combined_history_df)

combined_path = (
    LOCAL_RUN_DIR / "history_two_stage.csv"
)
combined_history_df.to_csv(
    combined_path,
    index=False,
)
trainer_b._sync_file(combined_path)

print("Stage B COMPLETE")


## 11. Acceleration-first Stage B summary / gate

최종 판단은 **Stage B만** 사용한다. Stage A는 feature-space adaptation 과정이므로
최종 checkpoint 후보와 섞지 않는다.

Stage B의 `best_accel_proxy.pt`가 overlap validation의 기본 후보이다.

In [ ]:
stage_b_df = combined_history_df[
    combined_history_df["phase"] == "stage_b"
].reset_index(drop=True)

if not len(stage_b_df):
    raise RuntimeError("empty Stage B history")

accel_col = "val/proxy/robust_mean_accel_macro_f1"
stage3_col = "val/proxy/robust_mean_stage3_score"
steer_col = "val/proxy/robust_mean_steer_macro_f1"
acc_cls_col = "val/proxy/medium/f1_accel_ACCELERATING"
d2c_col = "val/proxy/medium/dynamic_to_constant_rate"

idx = pd.to_numeric(
    stage_b_df[accel_col],
    errors="coerce",
).idxmax()
row = stage_b_df.loc[idx]

current = {
    "stage_b_epoch": int(row["epoch"]),
    "stage3": float(row[stage3_col]),
    "accel": float(row[accel_col]),
    "steer": float(row[steer_col]),
    "medium_accelerating": float(
        row[acc_cls_col]
    ),
    "medium_dynamic_to_constant": float(
        row[d2c_col]
    ),
}

delta = {
    key: current[key] - baseline_metrics[key]
    for key in (
        "stage3",
        "accel",
        "steer",
        "medium_accelerating",
        "medium_dynamic_to_constant",
    )
}

ac = vc["acceptance"]
checks = {
    "accel_plus_0p005": (
        delta["accel"]
        >= float(
            ac["min_accel_delta_vs_v5d_anchor"]
        )
    ),
    "accelerating_nonnegative": (
        delta["medium_accelerating"]
        >= float(
            ac["min_medium_accelerating_delta"]
        )
    ),
    "dynamic_to_constant_nonpositive": (
        delta["medium_dynamic_to_constant"]
        <= float(
            ac["max_dynamic_to_constant_delta"]
        )
    ),
}
local_accept = all(checks.values())

stage_a_last = history_a[-1] if history_a else None
stage_a_last_val = (
    stage_a_last.get("val")
    if stage_a_last is not None
    else None
)
if stage_a_last_val is None:
    stage_a_rows = combined_history_df[
        combined_history_df["phase"] == "stage_a"
    ]
    if len(stage_a_rows):
        last_a = stage_a_rows.iloc[-1]
        stage_a_last_val = {
            key[4:]: float(value)
            for key, value in last_a.items()
            if (
                isinstance(key, str)
                and key.startswith("val/")
                and pd.notna(value)
            )
        }

summary = {
    "run_variant": RUN_VARIANT,
    "git_commit": GIT_COMMIT,
    "videomamba_commit": ACTUAL_VM_COMMIT,
    "official_checkpoint": VM_CKPT_NAME,
    "backbone": "VideoMamba-S K400",
    "optimization": {
        "type": "two_stage",
        "effective_batch": (
            int(tc["batch_size"])
            * int(tc["grad_accum_steps"])
        ),
        "stage_a": {
            "epochs": int(stage_a_cfg["epochs"]),
            "max_steps_per_epoch": int(
                stage_a_cfg["max_steps_per_epoch"]
            ),
            "backbone": "fully_frozen",
        },
        "stage_b": {
            "epochs": int(stage_b_cfg["epochs"]),
            "max_steps_per_epoch": int(
                stage_b_cfg["max_steps_per_epoch"]
            ),
            "trainable_layers": list(
                ft_report.trainable_layer_indices
            ),
            "train_final_norm": bool(
                stage_b_cfg["train_final_norm"]
            ),
        },
    },
    "clip_len": int(dc["clip_len"]),
    "input_size": [
        int(dc["input_height"]),
        int(dc["input_width"]),
    ],
    "warm_start": {
        "run": cfg["warm_start"]["run_name"],
        "checkpoint": cfg["warm_start"]["checkpoint"],
        "copied_tensors": (
            warm_report["copied_tensors"]
            if warm_report is not None
            else None
        ),
    },
    "v5d_baseline": baseline_metrics,
    "stage_a_last_val": stage_a_last_val,
    "best_accel_stage_b_epoch": (
        current["stage_b_epoch"]
    ),
    "v6a_best_accel": current,
    "delta_vs_v5d": delta,
    "acceptance_checks": checks,
    "local_accept_for_overlap_validation": local_accept,
    "inference_lock": cfg["inference_lock"],
    "note": (
        "Final selection uses Stage B only. "
        "Local proxy is not DACON leaderboard. "
        "If accepted, run fixed V5-C overlap protocol "
        "without tuning state_weight."
    ),
}

local_summary = LOCAL_RUN_DIR / "summary.json"
local_summary.write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8",
)
trainer_b._sync_file(local_summary)

print(json.dumps(summary, indent=2))

if WANDB_ENABLED:
    finish_wandb()

print("Stage A latest             :", STAGE_A_DIR / "latest.pt")
print("Stage B latest             :", RUN_DIR / "latest.pt")
print("Stage B val-loss best      :", RUN_DIR / "best.pt")
print("Stage B proxy best         :", RUN_DIR / "best_proxy.pt")
print("Stage B accel-proxy best   :", RUN_DIR / "best_accel_proxy.pt")
print("next overlap validation?   :", local_accept)


## 다음 단계

`summary.json`과 `history_two_stage.csv`를 확인한다.

- `local_accept_for_overlap_validation = false`  
  → 빠른 VideoMamba-S recipe가 acceleration 병목을 해결하지 못한 것. M MASK 장시간 학습 전에 recipe를 재검토한다.

- `true`  
  → **Stage B `best_accel_proxy.pt`를 고정**하고 v5-C와 동일한 25 complete segments /
  T32 / stride8 / center_floor=.25 / production fusion / `state_weight=0`으로 overlap validation한다.
  S에서 유의미한 개선이 확인되면 같은 recipe를 최종 성능용 VideoMamba-M MASK에 승격한다.

공개 50 labels는 이 학습 notebook에서 사용하지 않는다.

S는 M보다 모델 크기/연산량이 훨씬 작아 실험 회전율을 높이기 위한 backbone이다.
custom BiMamba/causal-conv CUDA extension의 submission export/runtime 호환성 검증은 여전히 필요하다.
